<a href="https://colab.research.google.com/github/MrRichar02/Proyecto-Final-Modelos-2-G04/blob/main/desarrollo-proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importaciones generales

Asegurese de instalar las dependencias definidas para el proyecto en el `requirements.txt` o el `pyproject.toml`.

Si esta en colab solo necesita instalar la siguiente dependencia

In [1]:
!pip install ucimlrepo

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

## Obtención de dataset siguiendo la documentación de UC Irvine

In [3]:

# fetch dataset
online_shoppers_purchasing_intention_dataset = fetch_ucirepo(id=468)

# data (as pandas dataframes)
X = online_shoppers_purchasing_intention_dataset.data.features
y = online_shoppers_purchasing_intention_dataset.data.targets.copy()

online_shoppers_purchasing_intention_dataset['variables']

,name,role,type,demographic,description,units,missing_values
0,Administrative,Feature,Integer,None,None,None,no
1,Administrative_Duration,Feature,Integer,None,None,None,no
2,Informational,Feature,Integer,None,None,None,no
3,Informational_Duration,Feature,Integer,None,None,None,no
4,ProductRelated,Feature,Integer,None,None,None,no
5,ProductRelated_Duration,Feature,Continuous,None,None,None,no
6,BounceRates,Feature,Continuous,None,None,None,no
7,ExitRates,Feature,Continuous,None,None,None,no
8,PageValues,Feature,Integer,None,None,None,no
9,SpecialDay,Feature,Integer,None,None,None,no


In [4]:
X['Informational_Duration'].dtype

dtype('float64')

## Análisis variables

De acuerdo a la descripción del dataset disponible en UC Irvine, se concluye que las siguientes variables son categóricas.

- Revenue
- Weekend
- VisitorType
- TrafficType
- Region
- Browser
- OperatingSystems
- Month

In [5]:
vars_cat = ["Weekend","VisitorType","TrafficType","Region","Browser","OperatingSystems","Month"]
print("Revenue: ",np.unique(y["Revenue"]))
for i in vars_cat:
    print(i+": ", np.unique(X[i]))

Revenue:  [False  True]
Weekend:  [False  True]
VisitorType:  ['New_Visitor' 'Other' 'Returning_Visitor']
TrafficType:  [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
Region:  [1 2 3 4 5 6 7 8 9]
Browser:  [ 1  2  3  4  5  6  7  8  9 10 11 12 13]
OperatingSystems:  [1 2 3 4 5 6 7 8]
Month:  ['Aug' 'Dec' 'Feb' 'Jul' 'June' 'Mar' 'May' 'Nov' 'Oct' 'Sep']


## Descripción de las variables numéricas

In [6]:
num_cols = ['Administrative','Administrative_Duration','Informational',
            'Informational_Duration','ProductRelated','ProductRelated_Duration',
            'BounceRates','ExitRates','PageValues','SpecialDay']

In [7]:
for i in num_cols:
    print(i+": ", X[i].dtype)

Administrative:  int64
Administrative_Duration:  float64
Informational:  int64
Informational_Duration:  float64
ProductRelated:  int64
ProductRelated_Duration:  float64
BounceRates:  float64
ExitRates:  float64
PageValues:  float64
SpecialDay:  float64


In [8]:
X[num_cols].describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000


## Preparación encoding variables

### One-hot Encoding

In [9]:
X = pd.get_dummies(X, columns=['OperatingSystems'], prefix="OperatingSytem_", dtype=int)
X = pd.get_dummies(X, columns=['Browser'], prefix="Browser_", dtype=int)
X = pd.get_dummies(X, columns=['Region'], prefix="Region_", dtype=int)
X = pd.get_dummies(X, columns=['TrafficType'], prefix="TrafficType_", dtype=int)
X = pd.get_dummies(X, columns=['VisitorType'], prefix="VisitorType_", dtype=int)

### Label Encoding

In [10]:
X['Month'] = X['Month'].map({
    'Feb': 1,
    'Mar': 2,
    'May': 3,
    'June': 4,
    'Jul': 5,
    'Aug': 6,
    'Sep': 7,
    'Oct': 8,
    'Nov': 9,
    'Dec': 10
})

### Convertir valores tipos Boolean a int(0 y 1)

In [11]:
X['Weekend'] = X['Weekend'].astype(int)

In [12]:
y['Revenue'] = y['Revenue'].astype(int)

### Nuevos shapes

In [13]:
X.shape, y.shape


((12330, 65), (12330, 1))

## Separación de test y train y aplicación de SMOTE

### Dividimos el dataset en test y train, asignando un 20% para el test

In [14]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Aplicación de SMOTE para generar muestras artificiales de la clase minoritaria

#### Se obtiene la relación entre la clase positiva y la clase negativa

In [15]:
clase_negativo, clase_positivo = y_train.value_counts()
(clase_positivo)/(clase_negativo)

0.1830175101942912

### Primer porcentaje: Se aumenta la proporción de un 18% a un 20%

In [16]:
# Aplicar SMOTE SOLO al train
smote = SMOTE(random_state=42, sampling_strategy=0.2)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train,
    y_train
)

In [17]:
print(y_train.value_counts())
print(y_train_resampled.value_counts())

Revenue
0          8338
1          1526
Name: count, dtype: int64
Revenue
0          8338
1          1667
Name: count, dtype: int64


#### Segundo porcentaje: Se aumenta la proporción de un 18% a un 30%

In [18]:
# Aplicar SMOTE SOLO al train
smote1 = SMOTE(random_state=42, sampling_strategy=0.3)

X_train_resampled1, y_train_resampled1 = smote1.fit_resample(
    X_train,
    y_train
)

In [19]:
print(y_train.value_counts())
print(y_train_resampled1.value_counts())

Revenue
0          8338
1          1526
Name: count, dtype: int64
Revenue
0          8338
1          2501
Name: count, dtype: int64


#### Tercer porcentaje: Se aumenta la proporción de un 18% a un 40%

In [20]:
# Aplicar SMOTE SOLO al train
smote2 = SMOTE(random_state=42, sampling_strategy=0.4)

X_train_resampled2, y_train_resampled2 = smote2.fit_resample(
    X_train,
    y_train
)

In [21]:
print(y_train.value_counts())
print(y_train_resampled2.value_counts())

Revenue
0          8338
1          1526
Name: count, dtype: int64
Revenue
0          8338
1          3335
Name: count, dtype: int64


### Definición de metodología de validación

In [22]:
from sklearn.model_selection import StratifiedKFold

cv_strategy = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

## Entrenamiento de modelos usando pipeline con normalización, SMOTE y malla de hiperparametros

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.svm import SVC

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [24]:
y_train = y_train.values.ravel()

### Random Forest(Jenny)

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import numpy as np

In [ ]:
# PIPELINE  (scaler → SMOTE → RandomForest)
# El StandardScaler no es estrictamente necesario para RF, pero se mantiene para ser consistente con los demás modelos del proyecto y porque SMOTE opera mejor con datos escalados.

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy=0.3)),   # valor base; GridSearch lo variará
    ('classifier', RandomForestClassifier(
        random_state=42,
        n_jobs=-1,          # usar todos los núcleos disponibles
        class_weight='balanced'  # refuerzo adicional para clase minoritaria (Revenue=1)
    ))
])


In [ ]:
# Parámetros clave para RF:
#   n_estimators   → número de árboles; más árboles = más estable, más lento
#   max_depth      → profundidad máxima; None = crece hasta hojas puras (riesgo overfitting)
#   min_samples_split → muestras mínimas para dividir un nodo (regularización)
#   min_samples_leaf  → muestras mínimas en hoja (regularización)
#   max_features   → features evaluados en cada split; 'sqrt' es estándar para clasificación
#   sampling_strategy → proporción SMOTE (clase_positiva / clase_negativa)

param_grid_rf = {
    # SMOTE (evaluar los 3 porcentajes)
    'smote__sampling_strategy': [0.2, 0.3, 0.4],
    'smote__k_neighbors': [3, 5, 7],

    # RF — solo los más influyentes, valores fijos para el resto
    'classifier__n_estimators': [200],        # fijo, suficiente para estabilidad
    'classifier__max_depth': [10, 20],        # el más importante a explorar
    'classifier__min_samples_leaf': [1, 4],   # segundo más importante
    'classifier__max_features': ['sqrt'],     # fijo, es el estándar para clasificación
}

In [ ]:
# GRID SEARCH CON VALIDACIÓN CRUZADA ESTRATIFICADA
# Se usa roc_auc (igual que en la evaluación final del proyecto,
# ver: scoring = "roc_auc" y roc_auc_score en la sección de LR)

from sklearn.model_selection import StratifiedKFold

cv_strategy = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

grid_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    cv=cv_strategy,         # StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scoring='roc_auc',      # métrica principal del proyecto
    n_jobs=-1,
    verbose=2,
    refit=True              # refitea automáticamente con los mejores parámetros
)

In [ ]:
# Entrenamiento
grid_rf.fit(X_train, y_train)

Fitting 10 folds for each of 36 candidates, totalling 360 fits


GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote',
                                        SMOTE(random_state=42,
                                              sampling_strategy=0.3)),
                                       ('classifier',
                                        RandomForestClassifier(class_weight='balanced',
                                                               n_jobs=-1,
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [10, 20],
                         'classifier__max_features': ['sqrt'],
                         'classifier__min_samples_leaf': [1, 4],
                         'classifier__n_estimators': [200],
                         'smote__k_neighbors': [3, 5, 7],
                         'smote__sampling_strategy': [0.2, 0.3, 0.4]},
             scoring='roc_auc', verbose=2)

In [ ]:
# RESULTADOS DEL GRID SEARCH
print("=" * 60)
print("MEJORES HIPERPARÁMETROS ENCONTRADOS:")
print("=" * 60)
for param, value in grid_rf.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nMejor ROC-AUC (CV en train): {grid_rf.best_score_:.4f}")

MEJORES HIPERPARÁMETROS ENCONTRADOS:
  classifier__max_depth: 20
  classifier__max_features: sqrt
  classifier__min_samples_leaf: 4
  classifier__n_estimators: 200
  smote__k_neighbors: 7
  smote__sampling_strategy: 0.2

Mejor ROC-AUC (CV en train): 0.9289


In [ ]:
# VALIDACIÓN CRUZADA FINAL CON EL MEJOR PIPELINE
scores_rf = cross_val_score(
    grid_rf.best_estimator_,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring='roc_auc'
)

print(f"\nROC-AUC por fold (CV):")
print(np.round(scores_rf, 4))
print(f"Media:  {scores_rf.mean():.4f}")
print(f"Std:    {scores_rf.std():.4f}")


ROC-AUC por fold (CV):
[0.9267 0.9331 0.9152 0.9367 0.9094 0.9343 0.9389 0.9304 0.9421 0.9228]
Media:  0.9289
Std:    0.0100


In [ ]:
# EVALUACIÓN FINAL EN TEST
best_rf = grid_rf.best_estimator_
best_rf.fit(X_train, y_train)

y_pred_prob_rf = best_rf.predict_proba(X_test)[:, 1]
y_pred_rf = best_rf.predict(X_test)

roc_auc_test = roc_auc_score(y_test, y_pred_prob_rf)

print("\n" + "=" * 60)
print(f"ROC-AUC EN TEST: {roc_auc_test:.4f}")
print("=" * 60)

print("\nReporte de clasificación (threshold=0.5):")
print(classification_report(y_test, y_pred_rf, target_names=['No compra (0)', 'Compra (1)']))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_rf))



ROC-AUC EN TEST: 0.9180

Reporte de clasificación (threshold=0.5):
               precision    recall  f1-score   support

No compra (0)       0.95      0.91      0.93      2084
   Compra (1)       0.60      0.74      0.66       382

     accuracy                           0.88      2466
    macro avg       0.77      0.83      0.80      2466
 weighted avg       0.90      0.88      0.89      2466

Matriz de confusión:
[[1895  189]
 [  99  283]]


In [ ]:
#IMPORTANCIA DE FEATURES
import pandas as pd

feature_names = X_train.columns if hasattr(X_train, 'columns') else [f'f{i}' for i in range(X_train.shape[1])]

# Extraer el RF del pipeline final
rf_model = best_rf.named_steps['classifier']

importances = pd.Series(rf_model.feature_importances_, index=feature_names)
top_features = importances.sort_values(ascending=False).head(15)

print("\nTop 15 features más importantes:")
print(top_features.round(4).to_string())



Top 15 features más importantes:
PageValues                        0.4427
ExitRates                         0.0905
ProductRelated_Duration           0.0786
ProductRelated                    0.0605
Month                             0.0513
BounceRates                       0.0486
Administrative_Duration           0.0450
Administrative                    0.0354
Informational_Duration            0.0159
Informational                     0.0129
TrafficType__2                    0.0129
VisitorType__Returning_Visitor    0.0083
SpecialDay                        0.0077
VisitorType__New_Visitor          0.0077
Region__1                         0.0064


### XGBoost(Jenny)

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import numpy as np

In [ ]:
pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy=0.3)),
    ('classifier', XGBClassifier(
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',   # evita warning interno de XGBoost
        verbosity=0              # silencia logs innecesarios
    ))
])

In [ ]:
# MALLA DE HIPERPARÁMETROS

param_grid_xgb = {
    # SMOTE — los 3 porcentajes requeridos
    'smote__sampling_strategy': [0.2, 0.3, 0.4],
    'smote__k_neighbors': [3, 5, 7],

    # XGBoost — parámetros más influyentes
    'classifier__n_estimators': [200],
    'classifier__max_depth': [3, 6],          # profundidad del árbol base
    'classifier__learning_rate': [0.05, 0.1], # tasa de aprendizaje
    'classifier__subsample': [0.8],           # fracción de muestras por árbol
    'classifier__colsample_bytree': [0.8],    # fracción de features por árbol
}
# 3 × 3 × 1 × 2 × 2 × 1 × 1 = 36 combinaciones × 10 folds = 360 fits

In [ ]:
#GRID SEARCH
grid_xgb = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid_xgb,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2,
    refit=True
)

In [ ]:
grid_xgb.fit(X_train, y_train)

Fitting 10 folds for each of 36 candidates, totalling 360 fits


GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote',
                                        SMOTE(random_state=42,
                                              sampling_strategy=0.3)),
                                       ('classifier',
                                        XGBClassifier(base_score=None,
                                                      booster=None,
                                                      callbacks=None,
                                                      colsample_bylevel=None,
                                                      colsample_bynode=None,
                                                      colsample_bytree=None,
                                                      device=None,
                                                      early_stopping_r...
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=-1,
                                                      num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'classifier__colsample_bytree': [0.8],
                         'classifier__learning_rate': [0.05, 0.1],
                         'classifier__max_depth': [3, 6],
                         'classifier__n_estimators': [200],
                         'classifier__subsample': [0.8],
                         'smote__k_neighbors': [3, 5, 7],
                         'smote__sampling_strategy': [0.2, 0.3, 0.4]},
             scoring='roc_auc', verbose=2)

In [ ]:
# RESULTADOS
print("=" * 60)
print("MEJORES HIPERPARÁMETROS:")
print("=" * 60)
for param, value in grid_xgb.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nMejor ROC-AUC (CV en train): {grid_xgb.best_score_:.4f}")

MEJORES HIPERPARÁMETROS:
  classifier__colsample_bytree: 0.8
  classifier__learning_rate: 0.05
  classifier__max_depth: 6
  classifier__n_estimators: 200
  classifier__subsample: 0.8
  smote__k_neighbors: 5
  smote__sampling_strategy: 0.2

Mejor ROC-AUC (CV en train): 0.9345


In [ ]:
# EVALUACIÓN FINAL EN TEST

best_xgb = grid_xgb.best_estimator_
best_xgb.fit(X_train, y_train)

y_pred_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]
y_pred_xgb = best_xgb.predict(X_test)

roc_auc_test = roc_auc_score(y_test, y_pred_prob_xgb)

print("\n" + "=" * 60)
print(f"ROC-AUC EN TEST: {roc_auc_test:.4f}")
print("=" * 60)

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_xgb, target_names=['No compra (0)', 'Compra (1)']))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_xgb))


ROC-AUC EN TEST: 0.9279

Reporte de clasificación:
               precision    recall  f1-score   support

No compra (0)       0.93      0.95      0.94      2084
   Compra (1)       0.69      0.61      0.65       382

     accuracy                           0.90      2466
    macro avg       0.81      0.78      0.80      2466
 weighted avg       0.89      0.90      0.90      2466

Matriz de confusión:
[[1981  103]
 [ 148  234]]


### MLP (Santiago)

In [ ]:
pipeline_mlp = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', MLPClassifier(
        max_iter=1000,
        random_state=42
    ))
])

In [ ]:
param_grid_mlp = {

    'smote__sampling_strategy':[0.2,0.3],
    'smote__k_neighbors':[5],

    'classifier__hidden_layer_sizes':[
        (50,),
        (100,)
    ],

    'classifier__activation':[
        'relu'
    ],

    'classifier__alpha':[
        0.0001,
        0.001
    ],

    'classifier__learning_rate_init':[
        0.001,
        0.01
    ]
}

In [ ]:
grid_mlp = GridSearchCV(
    estimator=pipeline_mlp,
    param_grid=param_grid_mlp,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2
)

In [ ]:
grid_mlp.fit(X_train,y_train)

Fitting 10 folds for each of 16 candidates, totalling 160 fits


GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote', SMOTE(random_state=42)),
                                       ('classifier',
                                        MLPClassifier(max_iter=1000,
                                                      random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__activation': ['relu'],
                         'classifier__alpha': [0.0001, 0.001],
                         'classifier__hidden_layer_sizes': [(50,), (100,)],
                         'classifier__learning_rate_init': [0.001, 0.01],
                         'smote__k_neighbors': [5],
                         'smote__sampling_strategy': [0.2, 0.3]},
             scoring='roc_auc', verbose=2)

In [ ]:
grid_mlp.best_score_

np.float64(0.8683262488756853)

In [ ]:
print(grid_mlp.best_params_)

{'classifier__activation': 'relu', 'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.01, 'smote__k_neighbors': 5, 'smote__sampling_strategy': 0.2}


In [ ]:
best_mlp = grid_mlp.best_estimator_

best_mlp.fit(X_train,y_train)

y_pred_prob_mlp = best_mlp.predict_proba(X_test)[:,1]
y_pred_mlp = best_mlp.predict(X_test)

print("ROC-AUC TEST:")
print(roc_auc_score(y_test,y_pred_prob_mlp))

print(classification_report(y_test,y_pred_mlp))
print(confusion_matrix(y_test,y_pred_mlp))

ROC-AUC TEST:
0.8505768206529932
              precision    recall  f1-score   support

           0       0.92      0.92      0.92      2084
           1       0.55      0.54      0.55       382

    accuracy                           0.86      2466
   macro avg       0.73      0.73      0.73      2466
weighted avg       0.86      0.86      0.86      2466

[[1919  165]
 [ 177  205]]


### K-means(Santiago)


#### Implementación de K-Means como modelo de clasificación mediante asignación por clase mayoritaria

In [ ]:
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np

class KMeansClassifier(ClassifierMixin, BaseEstimator):

    _estimator_type = "classifier"

    def __init__(self, n_clusters=2, random_state=42):
        self.n_clusters = n_clusters
        self.random_state = random_state

    def fit(self, X, y):

        y = np.asarray(y).astype(int)

        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            random_state=self.random_state,
            n_init=10
        )

        clusters = self.kmeans_.fit_predict(X)

        self.cluster_labels_ = {}

        for c in np.unique(clusters):

            mask = (clusters == c)

            self.cluster_labels_[c] = np.bincount(
                y[mask]
            ).argmax()

        self.classes_ = np.array([0,1])

        return self

    def predict(self, X):

        clusters = self.kmeans_.predict(X)

        return np.array([
            self.cluster_labels_[c]
            for c in clusters
        ])

    def predict_proba(self, X):

        preds = self.predict(X)

        probs = np.zeros((len(preds),2))

        probs[:,0] = 1-preds
        probs[:,1] = preds

        return probs

In [ ]:
pipeline_kmeans = Pipeline([

    ('scaler', StandardScaler()),

    ('smote', SMOTE(random_state=42)),

    ('classifier', KMeansClassifier())
])

In [ ]:
param_grid_kmeans = {

    'smote__sampling_strategy': [
        0.2,
        0.3,
        0.4
    ],

    'smote__k_neighbors': [
        3,
        5,
        7
    ],

    'classifier__n_clusters': [
        2,
        3,
        4,
        5
    ]
}

In [ ]:
grid_kmeans = GridSearchCV(
    estimator=pipeline_kmeans,
    param_grid=param_grid_kmeans,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2,
    error_score='raise'
)

In [ ]:
grid_kmeans.fit(X_train,y_train)

Fitting 10 folds for each of 36 candidates, totalling 360 fits


GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             error_score='raise',
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote', SMOTE(random_state=42)),
                                       ('classifier', KMeansClassifier())]),
             n_jobs=-1,
             param_grid={'classifier__n_clusters': [2, 3, 4, 5],
                         'smote__k_neighbors': [3, 5, 7],
                         'smote__sampling_strategy': [0.2, 0.3, 0.4]},
             scoring='roc_auc', verbose=2)

In [ ]:
grid_kmeans.best_score_

np.float64(0.5)

In [ ]:
print(grid_kmeans.best_params_)

{'classifier__n_clusters': 2, 'smote__k_neighbors': 3, 'smote__sampling_strategy': 0.2}


In [ ]:
best_kmeans = grid_kmeans.best_estimator_

best_kmeans.fit(X_train,y_train)

y_pred_prob_kmeans = best_kmeans.predict_proba(X_test)[:,1]
y_pred_kmeans = best_kmeans.predict(X_test)

print("ROC-AUC TEST:")
print(
    roc_auc_score(
        y_test,
        y_pred_prob_kmeans
    )
)

print(
    classification_report(
        y_test,
        y_pred_kmeans
    )
)

print(
    confusion_matrix(
        y_test,
        y_pred_kmeans
    )
)

ROC-AUC TEST:
0.5
              precision    recall  f1-score   support

           0       0.85      1.00      0.92      2084
           1       0.00      0.00      0.00       382

    accuracy                           0.85      2466
   macro avg       0.42      0.50      0.46      2466
weighted avg       0.71      0.85      0.77      2466

[[2084    0]
 [ 382    0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### SVM(Ricardo)

In [ ]:
pipeline_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE()),
    ('svc', SVC())
])

In [ ]:
param_grid_svc = [
  {
      'smote__sampling_strategy': [0.2, 0.3],
      'svc__kernel': ['linear'],
      'svc__C': [0.01, 0.1, 1]
  },
  {
      'smote__sampling_strategy': [0.2, 0.3],
      'svc__kernel': ['rbf'],
      'svc__C': [0.01, 0.1, 1],
      'svc__gamma': ['scale', 'auto', 0.01, 0.1],
  }

]

In [ ]:
grid_svc = GridSearchCV(
    estimator=pipeline_svc,
    param_grid=param_grid_svc,
    cv=cv_strategy,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

In [ ]:
grid_svc.fit(X_train,y_train)

Fitting 10 folds for each of 30 candidates, totalling 300 fits


GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote', SMOTE()), ('svc', SVC())]),
             n_jobs=-1,
             param_grid=[{'smote__sampling_strategy': [0.2, 0.3],
                          'svc__C': [0.01, 0.1, 1], 'svc__kernel': ['linear']},
                         {'smote__sampling_strategy': [0.2, 0.3],
                          'svc__C': [0.01, 0.1, 1],
                          'svc__gamma': ['scale', 'auto', 0.01, 0.1],
                          'svc__kernel': ['rbf']}],
             scoring='f1', verbose=2)

In [ ]:
grid_svc.best_score_

np.float64(0.6100854005716715)

In [ ]:
# RESULTADOS
print("=" * 60)
print("MEJORES HIPERPARÁMETROS:")
print("=" * 60)
for param, value in grid_svc.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nMejor F1 (CV en train): {grid_svc.best_score_:.4f}")

MEJORES HIPERPARÁMETROS:
  smote__sampling_strategy: 0.3
  svc__C: 1
  svc__kernel: linear

Mejor F1 (CV en train): 0.6101


In [ ]:
# EVALUACIÓN FINAL EN TEST

best_svc = grid_svc.best_estimator_
best_svc.fit(X_train, y_train)

#y_pred_prob_svc = best_svc.predict_proba(X_test)[:, 1]
y_pred_svc = best_svc.predict(X_test)

#roc_auc_test = roc_auc_score(y_test, y_pred_prob_svc)

#print("\n" + "=" * 60)
#print(f"ROC-AUC EN TEST: {roc_auc_test:.4f}")
#print("=" * 60)

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_svc, target_names=['No compra (0)', 'Compra (1)']))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_svc))


Reporte de clasificación:
               precision    recall  f1-score   support

No compra (0)       0.91      0.95      0.93      2084
   Compra (1)       0.66      0.51      0.57       382

     accuracy                           0.88      2466
    macro avg       0.79      0.73      0.75      2466
 weighted avg       0.87      0.88      0.88      2466

Matriz de confusión:
[[1984  100]
 [ 189  193]]


### GMM(Ricardo)

In [28]:
import numpy as np

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.mixture import GaussianMixture
from sklearn.utils.validation import check_X_y, check_array


class GMMClassifier(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        n_components=1,
        covariance_type='full',
        max_iter=100,
        n_init=1,
        init_params='kmeans',
        random_state=None
    ):
        self.n_components = n_components
        self.covariance_type = covariance_type
        self.max_iter = max_iter
        self.n_init = n_init
        self.init_params = init_params
        self.random_state = random_state

    def fit(self, X, y):

        X, y = check_X_y(X, y)

        self.classes_ = np.unique(y)
        self.models_ = {}

        self.priors_ = {}

        for c in self.classes_:

            gmm = GaussianMixture(
                n_components=self.n_components,
                covariance_type=self.covariance_type,
                max_iter=self.max_iter,
                n_init=self.n_init,
                init_params=self.init_params,
                random_state=self.random_state
            )

            X_class = X[y == c]

            gmm.fit(X_class)

            self.models_[c] = gmm

            # Prior de clase
            self.priors_[c] = X_class.shape[0] / X.shape[0]

        return self

    def predict_proba(self, X):

        X = check_array(X)

        log_probs = np.zeros((X.shape[0], len(self.classes_)))

        for idx, c in enumerate(self.classes_):

            gmm = self.models_[c]

            # log p(x | clase)
            log_likelihood = gmm.score_samples(X)

            # log p(clase)
            log_prior = np.log(self.priors_[c])

            log_probs[:, idx] = log_likelihood + log_prior

        # estabilidad numérica
        log_probs -= log_probs.max(axis=1, keepdims=True)

        probs = np.exp(log_probs)

        probs /= probs.sum(axis=1, keepdims=True)

        return probs

    def predict(self, X):

        probs = self.predict_proba(X)

        return self.classes_[np.argmax(probs, axis=1)]


In [40]:
pipeline_gmm = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE()),
    ('gmm', GMMClassifier(
        random_state=42
    ))
])

In [41]:
param_grid_gmm = {
    'smote__sampling_strategy': [0.2, 0.3, 0.4],
    'gmm__n_components': [1, 2, 3, 4],
    'gmm__covariance_type': ['full', 'diag', 'tied', 'spherical'],
    'gmm__max_iter': [100, 200],
    'gmm__n_init': [1, 2],
    'gmm__init_params': ['kmeans', 'k-means++', 'random', 'random_from_data'],
}

In [46]:
grid_gmm = GridSearchCV(
    estimator=pipeline_gmm,
    param_grid=param_grid_gmm,
    cv=cv_strategy,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

In [48]:
grid_gmm.fit(X_train,y_train)

Fitting 10 folds for each of 768 candidates, totalling 7680 fits


GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote', SMOTE()),
                                       ('gmm',
                                        GMMClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'gmm__covariance_type': ['full', 'diag', 'tied',
                                                  'spherical'],
                         'gmm__init_params': ['kmeans', 'k-means++', 'random',
                                              'random_from_data'],
                         'gmm__max_iter': [100, 200],
                         'gmm__n_components': [1, 2, 3, 4],
                         'gmm__n_init': [1, 2],
                         'smote__sampling_strategy': [0.2, 0.3, 0.4]},
             scoring='f1', verbose=2)

In [49]:
grid_gmm.best_score_

np.float64(0.5016500428346755)

In [50]:
grid_gmm.best_params_

{'gmm__covariance_type': 'spherical',
 'gmm__init_params': 'random_from_data',
 'gmm__max_iter': 200,
 'gmm__n_components': 1,
 'gmm__n_init': 1,
 'smote__sampling_strategy': 0.3}

In [51]:
# EVALUACIÓN FINAL EN TEST

best_gmm = grid_gmm.best_estimator_
best_gmm.fit(X_train, y_train)

#y_pred_prob_gmm = best_gmm.predict_proba(X_test)[:, 1]
y_pred_gmm = best_gmm.predict(X_test)

#roc_auc_test = roc_auc_score(y_test, y_pred_prob_gmm)

#print("\n" + "=" * 60)
#print(f"ROC-AUC EN TEST: {roc_auc_test:.4f}")
#print("=" * 60)

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_gmm, target_names=['No compra (0)', 'Compra (1)']))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_gmm))


Reporte de clasificación:
               precision    recall  f1-score   support

No compra (0)       0.91      0.89      0.90      2084
   Compra (1)       0.45      0.52      0.48       382

     accuracy                           0.83      2466
    macro avg       0.68      0.70      0.69      2466
 weighted avg       0.84      0.83      0.83      2466

Matriz de confusión:
[[1846  238]
 [ 185  197]]


### Creación de pipeline

In [ ]:
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', smote),
    ('logisticRegression', LogisticRegression(
        max_iter=1000,
        random_state=42,
    ))
])

### Definición de las métricas

In [ ]:
#scoring = ['roc_auc','average_precision', 'f1']

In [ ]:
scoring = "roc_auc"

### Aplicación de la metodología de validación

In [ ]:
y_train = y_train.values.ravel()

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipeline_lr,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=scoring
)

In [ ]:
scores

array([0.87621667, 0.89047194, 0.883215  , 0.89449225, 0.86162912,
       0.9083286 , 0.90599363, 0.88650133, 0.90881058, 0.90555438])

### Evaluación final en test

In [ ]:
pipeline_lr.fit(X_train, y_train)

y_pred_prob = pipeline_lr.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, y_pred_prob)

np.float64(0.8671528775713238)